# Biohub Cell Tracking Exp006 UNet Edge 065

Sanitized Kaggle submission notebook for Biohub cell tracking. It runs UNet + transformer + ILP inference from public artifacts and writes `/kaggle/working/submission.csv`.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ model -- #
COMPETITION     = "biohub-cell-tracking-during-development"
METHOD          = "unet_transformer"
UNET_BATCH      = int(os.environ.get("BIOHUB_UNET_BATCH", "4"))
ILP_EDGE_W      = float(os.environ.get("BIOHUB_ILP_EDGE_W", "-0.65"))
ILP_APPEAR_W    = float(os.environ.get("BIOHUB_ILP_APPEAR_W", "0.1"))
ILP_DISAPPEAR_W = float(os.environ.get("BIOHUB_ILP_DISAPPEAR_W", "0.1"))

# ---------------------------------------------------------------- experiment -- #
# Presets:
#   geom_prune : default. Reproduce the strong ~0.81-style ILP setting, then
#                apply mild geometry-based edge/division cleanup after inference.
#   raw081     : no post-processing; reproduce the previous 0.81-style run.
#   cv         : run the small CV sweep and then apply the same geometry cleanup.
PRESET = os.environ.get("BIOHUB_PRESET", "geom_prune").lower()

# Set BIOHUB_PUBLIC_SCORE after submission, rerun only the final cells, and the

# ---------------------------------------------------------- post-processing -- #
# This notebook explores a different direction from the div-weight-only run:
# keep the strong ILP inference, then remove only suspicious edges afterwards.
GEOM_CLEAN_EDGES = os.environ.get("BIOHUB_GEOM_CLEAN_EDGES", "1").lower() in {"1", "true", "yes"}
DROP_LONG_EDGES  = os.environ.get("BIOHUB_DROP_LONG_EDGES", "1").lower() in {"1", "true", "yes"}
MAX_EDGE_JUMP_UM = float(os.environ.get("BIOHUB_MAX_EDGE_JUMP_UM", "18.0"))

PRUNE_DIVISION_GEOMETRY = os.environ.get("BIOHUB_PRUNE_DIVISION_GEOMETRY", "1").lower() in {"1", "true", "yes"}
DIV_KEEP_TOP_K          = int(os.environ.get("BIOHUB_DIV_KEEP_TOP_K", "2"))
DIV_MIN_DAUGHTER_SEP_UM = float(os.environ.get("BIOHUB_DIV_MIN_DAUGHTER_SEP_UM", "1.8"))
DIV_MAX_CHILD_JUMP_UM   = float(os.environ.get("BIOHUB_DIV_MAX_CHILD_JUMP_UM", "10.0"))
DIV_DISTANCE_RATIO_MAX  = float(os.environ.get("BIOHUB_DIV_DISTANCE_RATIO_MAX", "2.4"))

# -------------------------------------------------------------------- CV -- #
# Default is no CV because the previous run already showed the best useful
# detector region. This keeps the second parallel run much faster.
RUN_CV = os.environ.get("BIOHUB_RUN_CV", "0").lower() in {"1", "true", "yes"}

CV_DET_THRESHOLDS = [
    float(x) for x in os.environ.get(
        "BIOHUB_CV_DET_THRESHOLDS",
        "0.995,0.997,0.99"
    ).split(",") if x.strip()
]
CV_DIV_WEIGHTS = [
    float(x) for x in os.environ.get(
        "BIOHUB_CV_DIV_WEIGHTS",
        "0.8,1.0"
    ).split(",") if x.strip()
]
CV_MAX_EMBRYOS = int(os.environ.get("BIOHUB_CV_MAX_EMBRYOS", "2"))
CV_TIE_TOL = float(os.environ.get("BIOHUB_CV_TIE_TOL", "0.0015"))

if PRESET == "raw081":
    BEST_DET_THR = 0.995
    BEST_DIV_W   = 0.8
    RUN_CV       = False
    GEOM_CLEAN_EDGES = False
elif PRESET == "cv":
    BEST_DET_THR = 0.995
    BEST_DIV_W   = 0.8
    RUN_CV       = True
else:
    # Default: keep the previous ~0.81 inference setting and improve via
    # post-inference graph cleanup rather than changing ILP weights.
    BEST_DET_THR = 0.995
    BEST_DIV_W   = 0.8

if os.environ.get("BIOHUB_FORCE_DET_THR"):
    BEST_DET_THR = float(os.environ["BIOHUB_FORCE_DET_THR"])
if os.environ.get("BIOHUB_FORCE_DIV_W"):
    BEST_DIV_W = float(os.environ["BIOHUB_FORCE_DIV_W"])
# ----------------------------------------------------------------- paths -- #
COMP_DIR = next(
    (
        p for p in [
            Path(f"/kaggle/input/competitions/{COMPETITION}"),
            Path(f"/kaggle/input/{COMPETITION}"),
        ]
        if p.exists()
    ),
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
)
TEST_DIR  = COMP_DIR / "test"
TRAIN_DIR = COMP_DIR / "train"
WORKING   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR  = WORKING / "cellmot_repo"
SUB_PATH  = WORKING / "submission.csv"

print(f"PRESET   : {PRESET}")
print(f"RUN_CV   : {RUN_CV}")
print(f"COMP_DIR : {COMP_DIR}  (exists={COMP_DIR.exists()})")
print(f"TEST_DIR : {TEST_DIR}  (exists={TEST_DIR.exists()})")
print(f"TRAIN_DIR: {TRAIN_DIR} (exists={TRAIN_DIR.exists()})")
print(f"Initial params: det_thr={BEST_DET_THR}, div_w={BEST_DIV_W}")
print(f"Postprocess: geom_clean={GEOM_CLEAN_EDGES}, drop_long={DROP_LONG_EDGES}, max_jump_um={MAX_EDGE_JUMP_UM}, prune_div_geom={PRUNE_DIVISION_GEOMETRY}")
print(f"Division geom: keep_top_k={DIV_KEEP_TOP_K}, min_daughter_sep_um={DIV_MIN_DAUGHTER_SEP_UM}, max_child_jump_um={DIV_MAX_CHILD_JUMP_UM}, ratio_max={DIV_DISTANCE_RATIO_MAX}")
print(f"CV grid: thresholds={CV_DET_THRESHOLDS}, div_weights={CV_DIV_WEIGHTS}, max_embryos={CV_MAX_EMBRYOS}, tie_tol={CV_TIE_TOL}")

try:
    import torch
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("GPU name :", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch/GPU audit skipped:", repr(exc))


## Offline setup

Everything — wheels, repository source, pretrained weights — comes from the `cellmot-baseline-artifacts` public dataset.  No internet needed.

In [ ]:
def find_artifacts() -> Path:
    """Find the external artifact dataset that contains repo/, weights/, and wheels/.

    Required Kaggle input dataset:
        cellmot-baseline-artifacts

    The exact mount path can vary depending on whether the notebook is run as a
    fork, a copied notebook, or with a dataset owner namespace. Therefore this
    finder checks common locations first and then performs a shallow recursive
    scan under /kaggle/input.
    """
    root = Path("/kaggle/input")
    candidates: list[Path] = [
        Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts"),
    ]

    if root.exists():
        # First inspect direct children, then one additional level. This is fast
        # and catches most Kaggle input mounting layouts.
        for child in sorted(root.iterdir()):
            if child.is_dir():
                candidates.append(child)
                candidates.append(child / child.name)
                if "cellmot" in child.name.lower() or "artifact" in child.name.lower():
                    for sub in sorted(child.iterdir()) if child.exists() else []:
                        if sub.is_dir():
                            candidates.append(sub)

        # Last resort: bounded recursive search for the exact required layout.
        for repo_dir in root.glob("**/repo"):
            p = repo_dir.parent
            candidates.append(p)

    seen: set[Path] = set()
    checked: list[str] = []
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if p in seen:
            continue
        seen.add(p)
        checked.append(str(p))
        if (p / "repo").exists() and (p / "weights").exists() and (p / "wheels").exists():
            return p

    input_dirs = []
    if root.exists():
        input_dirs = sorted(str(p) for p in root.iterdir() if p.is_dir())

    message = f"""
cellmot-baseline-artifacts was not found.

This UNet + ILP notebook cannot run from the competition data alone. It needs an
additional Kaggle Dataset containing:

    repo/
    weights/
    wheels/

Please add the following dataset to Notebook > Input:

    cellmot-baseline-artifacts

Known/expected mount paths include:

    /kaggle/input/cellmot-baseline-artifacts
    /kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts

Currently visible /kaggle/input directories:
{chr(10).join('  - ' + x for x in input_dirs[:30]) if input_dirs else '  - none'}

Checked candidate paths:
{chr(10).join('  - ' + x for x in checked[:40])}

After adding the artifact dataset, rerun the notebook with GPU enabled.
""".strip()
    raise FileNotFoundError(message)


ARTIFACTS = find_artifacts()
print("ARTIFACTS:", ARTIFACTS)
print("Contents: ", sorted(p.name for p in ARTIFACTS.iterdir()))

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", str(ARTIFACTS / "wheels"),
        "--quiet",
        "tracksdata", "zarr>=3.0.10", "pyscipopt",
    ],
    check=True,
)
print("Packages installed from artifact wheels")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)
sys.path.insert(0, str(REPO_DIR / "src"))

weights_root = REPO_DIR / "weights" / METHOD
AVAILABLE_SPLITS = (
    sorted(d.name for d in weights_root.iterdir() if d.is_dir())
    if weights_root.exists() else []
)
if not AVAILABLE_SPLITS:
    raise FileNotFoundError(f"No model splits found under {weights_root}")

print(f"Available model splits: {AVAILABLE_SPLITS}")
for sp in AVAILABLE_SPLITS:
    wd = REPO_DIR / "weights" / METHOD / sp
    print(f"  {sp}: {sorted(p.name for p in wd.iterdir())}")


## Cross-validation framework

The proxy metric isn't a perfect copy of the competition score — I don't have the exact over-prediction penalty formula — but it captures the same structure:

- Nodes are matched per-frame with a Hungarian assignment gated at 7 µm (physical coordinates, z=1.625 µm/vox, y=x=0.40625 µm/vox).
- Edge Jaccard is computed over the matched subgraph.  When `N_pred > estimated_N_gt`, it's multiplied by `N_est / N_pred` — a conservative stand-in for the official penalty.
- Division Jaccard: nodes with ≥2 outgoing edges are division events.  After mapping predicted divisions through the node-match, TP/FP/FN give a per-sample Jaccard.
- Combined = mean(adj_ej, div_j) — rough approximation, but enough to rank settings.

Training samples are drawn one-per-embryo to keep the CV embryo-diverse, matching the train/test disjointness structure.

In [ ]:
import zarr
import tracksdata as td
from scipy.optimize import linear_sum_assignment

SCALE_UM = np.array([1.625, 0.40625, 0.40625])   # z, y, x µm/voxel
GATE_UM  = 7.0


def list_zarr_names(directory: Path) -> list[str]:
    if not directory.exists():
        return []
    return sorted(p.stem for p in directory.iterdir() if p.suffix == ".zarr")


def read_geff(path: Path):
    """Returns (nodes_df, edges_df, estimated_total_nodes)."""
    g    = zarr.open_group(str(path), mode="r")
    ids  = np.asarray(g["nodes/ids"][:]).astype(np.int64)
    props = {k: np.asarray(g[f"nodes/props/{k}/values"][:]) for k in ("t","z","y","x")}
    nodes = pd.DataFrame({"node_id": ids, **props})

    raw = np.asarray(g["edges/ids"][:]).astype(np.int64)
    edges = (
        pd.DataFrame(raw, columns=["source_id", "target_id"])
        if raw.ndim == 2 and raw.shape[1] == 2
        else pd.DataFrame(columns=["source_id", "target_id"])
    )

    est = np.nan
    try:
        meta = json.loads((path / "zarr.json").read_text())

        def _dig(obj):
            if isinstance(obj, dict):
                if "estimated_number_of_nodes" in obj:
                    return obj["estimated_number_of_nodes"]
                for v in obj.values():
                    r = _dig(v)
                    if r is not None:
                        return r
            return None

        v = _dig(meta)
        if v is not None:
            est = float(v)
    except Exception:
        pass

    return nodes, edges, est


def load_pred_graph(geff_path: Path):
    result = td.graph.IndexedRXGraph.from_geff(str(geff_path))
    return result[0] if isinstance(result, tuple) else result


def graph_to_rows(graph, dataset: str) -> list[dict]:
    rows: list[dict] = []
    for r in graph.node_attrs().iter_rows(named=True):
        rows.append({
            "dataset": dataset, "row_type": "node",
            "node_id": int(r["node_id"]),
            "t": int(r["t"]),
            "z": int(round(float(r["z"]))),
            "y": int(round(float(r["y"]))),
            "x": int(round(float(r["x"]))),
            "source_id": -1, "target_id": -1,
        })
    for r in graph.edge_attrs().iter_rows(named=True):
        rows.append({
            "dataset": dataset, "row_type": "edge",
            "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(r["source_id"]),
            "target_id": int(r["target_id"]),
        })
    return rows


def proxy_score(
    pn: pd.DataFrame,
    pe: pd.DataFrame,
    gn: pd.DataFrame,
    ge: pd.DataFrame,
    n_est: float = np.nan,
) -> dict:
    # --- node matching per timepoint ---
    p2g: dict[int, int] = {}
    for t in sorted(set(pn["t"].unique()) & set(gn["t"].unique())):
        p_t = pn[pn["t"] == t].reset_index(drop=True)
        g_t = gn[gn["t"] == t].reset_index(drop=True)
        if len(p_t) == 0 or len(g_t) == 0:
            continue
        D = np.sqrt(
            ((p_t[["z","y","x"]].values * SCALE_UM)[:, None] -
             (g_t[["z","y","x"]].values * SCALE_UM)[None]) ** 2
        ).sum(2)
        cost = np.where(D <= GATE_UM, D, 1e9)
        ri, ci = linear_sum_assignment(cost)
        for r, c in zip(ri, ci):
            if cost[r, c] < 1e9:
                p2g[int(p_t.loc[r, "node_id"])] = int(g_t.loc[c, "node_id"])

    # --- edge Jaccard ---
    gt_edge_set = set(
        zip(ge["source_id"].astype(int), ge["target_id"].astype(int))
    )
    pm = {
        (p2g[int(s)], p2g[int(t)])
        for s, t in zip(pe["source_id"].astype(int), pe["target_id"].astype(int))
        if int(s) in p2g and int(t) in p2g
    }
    e_tp  = len(pm & gt_edge_set)
    e_fp  = len(pm) - e_tp
    e_fn  = len(gt_edge_set) - e_tp
    raw_ej = e_tp / max(e_tp + e_fp + e_fn, 1)

    n_pred = len(pn)
    penalty = (
        min(1.0, float(n_est) / max(n_pred, 1))
        if np.isfinite(n_est) and n_pred > n_est
        else 1.0
    )
    adj_ej = raw_ej * penalty

    # --- division Jaccard ---
    def _div_nodes(e: pd.DataFrame) -> set[int]:
        if len(e) == 0:
            return set()
        cnt = e.groupby("source_id").size()
        return set(cnt[cnt >= 2].index.astype(int))

    gt_divs  = _div_nodes(ge)
    pd_divs  = _div_nodes(pe)
    pd_divs_mapped = {p2g[n] for n in pd_divs if n in p2g}

    d_tp = len(pd_divs_mapped & gt_divs)
    d_fp = len(pd_divs) - d_tp
    d_fn = len(gt_divs) - d_tp
    div_j = d_tp / max(d_tp + d_fp + d_fn, 1)

    return {
        "n_pred": n_pred, "n_gt": len(gn), "n_est": n_est,
        "n_matched": len(p2g), "penalty": round(penalty, 4),
        "raw_ej": round(raw_ej, 4), "adj_ej": round(adj_ej, 4),
        "n_gt_div": len(gt_divs), "n_pred_div": len(pd_divs),
        "div_tp": d_tp, "div_j": round(div_j, 4),
        "combined": round((adj_ej + div_j) / 2, 4),
    }


print("Utilities ready")

In [ ]:
def pick_cv_samples(train_dir: Path, n: int) -> list[str]:
    names = [
        nm for nm in list_zarr_names(train_dir)
        if (train_dir / (nm + ".geff")).exists()
    ]
    seen: set[str] = set()
    picked: list[str] = []
    for nm in names:
        eid = nm.split("_")[0]
        if eid not in seen:
            seen.add(eid)
            picked.append(nm)
        if len(picked) >= n:
            break
    return picked


cv_samples = pick_cv_samples(TRAIN_DIR, CV_MAX_EMBRYOS)
print(f"CV samples ({len(cv_samples)}): {cv_samples}")

if not cv_samples:
    print("WARNING: no CV samples found — will skip sweep and use defaults.")

In [ ]:
def run_predict(
    data_dir: Path,
    sample_names: list[str],
    det_thr: float,
    div_w: float,
    split_name: str = "split_0",
    tag: str = "run",
) -> list[Path]:
    """Invoke predict_unet_transformer.py; return sorted list of output .geff paths."""
    splits_file = REPO_DIR / f"splits_{tag}.json"
    splits_file.write_text(
        json.dumps([{"split": 0, "train": [], "test": sample_names}])
    )

    # wipe previous predictions so runs don't bleed into each other
    pred_base = REPO_DIR / "predictions"
    if pred_base.exists():
        shutil.rmtree(pred_base)

    cmd = [
        sys.executable, "scripts/predict_unet_transformer.py",
        "--data-dir",                str(data_dir),
        "--splits",                  splits_file.name,
        "--split",                   "0",
        "--weights",                 f"weights/{METHOD}/{split_name}/edge_predictor_best.pth",
        "--unet-batch-size",         str(UNET_BATCH),
        "--det-threshold",           str(det_thr),
        "--ilp-edge-weight",         str(ILP_EDGE_W),
        "--ilp-appearance-weight",   str(ILP_APPEAR_W),
        "--ilp-disappearance-weight",str(ILP_DISAPPEAR_W),
        "--ilp-division-weight",     str(div_w),
        "--use-ilp",
    ]

    result = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": "src"},
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Prediction script failed (thr={det_thr}, div_w={div_w}):\n"
            + result.stderr.decode(errors="replace")[:3000]
        )

    found = sorted(pred_base.rglob("*.geff")) if pred_base.exists() else []
    if not found:
        raise RuntimeError(f"No .geff outputs found under {pred_base}")
    return found


print("run_predict helper ready")

In [ ]:
cv_records: list[dict] = []
cv_start = time.time()

if RUN_CV and cv_samples:
    combos = list(product(CV_DET_THRESHOLDS, CV_DIV_WEIGHTS))
    print(
        f"{len(combos)} combos × {len(cv_samples)} samples "
        f"= {len(combos)*len(cv_samples)} total predictions"
    )

    for det_thr, div_w in combos:
        print(f"\n>>> det_thr={det_thr}  div_w={div_w}")
        t0 = time.time()
        try:
            pred_geffs = run_predict(
                TRAIN_DIR, cv_samples, det_thr, div_w,
                tag=f"cv_{str(det_thr).replace('.','p')}_{str(div_w).replace('.','p')}",
            )
            combo_c: list[float] = []
            for geff_path in pred_geffs:
                stem   = geff_path.stem
                gt_pth = TRAIN_DIR / (stem + ".geff")
                if not gt_pth.exists():
                    print(f"    [skip] no GT for {stem}")
                    continue
                gn, ge, n_est = read_geff(gt_pth)
                graph = load_pred_graph(geff_path)
                rows  = pd.DataFrame(graph_to_rows(graph, stem))
                pn    = rows[rows["row_type"] == "node"]
                pe    = rows[rows["row_type"] == "edge"]
                sc    = proxy_score(pn, pe, gn, ge, n_est)
                sc.update(dataset=stem, det_thr=det_thr, div_w=div_w)
                cv_records.append(sc)
                combo_c.append(sc["combined"])
                print(
                    f"    {stem[:28]:28s}  "
                    f"adj_ej={sc['adj_ej']:.4f}  "
                    f"div_j={sc['div_j']:.4f}  "
                    f"combined={sc['combined']:.4f}  "
                    f"pred_div={sc['n_pred_div']}  gt_div={sc['n_gt_div']}"
                )
            m = np.mean(combo_c) if combo_c else float("nan")
            print(f"    => mean combined = {m:.4f}  ({time.time()-t0:.0f}s)")
        except Exception as exc:
            print(f"    FAILED: {str(exc)[:250]}")

    print(f"\nCV wall time: {(time.time()-cv_start)/60:.1f} min")
else:
    print("CV sweep skipped. Using preset/default parameters.")

In [ ]:
if cv_records:
    cv_df = pd.DataFrame(cv_records)
    summary = (
        cv_df.groupby(["det_thr", "div_w"])
        .agg(
            adj_ej=("adj_ej", "mean"),
            div_j=("div_j", "mean"),
            combined=("combined", "mean"),
            pred_div=("n_pred_div", "mean"),
            matched=("n_matched", "mean"),
            n_pred=("n_pred", "mean"),
        )
        .round(4)
        .sort_values("combined", ascending=False)
    )
    print("Mean proxy scores across CV embryos:")
    display(summary)

    best_combined = float(summary["combined"].max())
    near_best = summary[summary["combined"] >= best_combined - CV_TIE_TOL].copy()

    # Conservative tie-break:
    # If proxy scores are nearly tied, prefer fewer predicted division-like
    # sources. For this geometry-cleanup notebook, proxy ties still prefer fewer divisions,
    # but the default run skips CV and uses raw081 parameters plus post-processing.
    near_best = near_best.sort_values(
        ["pred_div", "combined", "adj_ej"],
        ascending=[True, False, False],
    )
    best_idx = near_best.index[0]
    BEST_DET_THR = float(best_idx[0])
    BEST_DIV_W   = float(best_idx[1])

    selected = summary.loc[best_idx]
    base_c = float(
        summary.loc[(0.995, 0.8), "combined"]
        if (0.995, 0.8) in summary.index
        else float("nan")
    )

    print(f"\nBest proxy combined      : {best_combined:.4f}")
    print(f"Selected with tie-break  : det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}")
    print(f"Selected combined        : {float(selected['combined']):.4f}")
    print(f"Selected avg pred_div    : {float(selected['pred_div']):.1f}")
    if np.isfinite(base_c):
        print(f"vs previous 0.81-style 0.995/0.8: combined={base_c:.4f}  "
              f"delta={float(selected['combined']) - base_c:+.4f}")
else:
    print(f"No CV data. Using preset/default params: det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}")

print(f"\nFinal inference params: det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}")


## Test inference and graph post-processing

This version intentionally keeps the strong 0.81-style inference parameters by default.  The separate score-up idea is applied after inference:

- Remove non-forward or very long temporal edges.
- Cap excessive outgoing edges from one source.
- Demote geometrically implausible division-like nodes to single continuations.

This provides a clean A/B against the division-weight experiment because node detection and raw ILP settings stay close to the previous 0.81 run.


In [ ]:

def _physical_edge_table(df: pd.DataFrame, ds: str) -> pd.DataFrame:
    """Return edge rows joined with source/target coordinates for one dataset."""
    g = df[df["dataset"] == ds]
    nodes = g[g["row_type"] == "node"][["node_id", "t", "z", "y", "x"]].copy()
    edges = g[g["row_type"] == "edge"][["source_id", "target_id"]].copy()
    if len(nodes) == 0 or len(edges) == 0:
        return pd.DataFrame()

    edges = edges.reset_index().rename(columns={"index": "edge_index"})
    src = nodes.rename(columns={
        "node_id": "source_id", "t": "s_t", "z": "s_z", "y": "s_y", "x": "s_x"
    })
    tgt = nodes.rename(columns={
        "node_id": "target_id", "t": "t_t", "z": "t_z", "y": "t_y", "x": "t_x"
    })
    e = edges.merge(src, on="source_id", how="left").merge(tgt, on="target_id", how="left")
    coord_cols = ["s_t", "s_z", "s_y", "s_x", "t_t", "t_z", "t_y", "t_x"]
    e = e.dropna(subset=coord_cols).copy()
    if len(e) == 0:
        return e
    e["dt"] = e["t_t"] - e["s_t"]
    dz = (e["t_z"] - e["s_z"]) * SCALE_UM[0]
    dy = (e["t_y"] - e["s_y"]) * SCALE_UM[1]
    dx = (e["t_x"] - e["s_x"]) * SCALE_UM[2]
    e["jump_um"] = np.sqrt(dz * dz + dy * dy + dx * dx)
    return e


def geometry_cleanup_submission(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Mild post-ILP cleanup for suspicious temporal edges and divisions.

    This does not change node detections.  It only drops edges that are likely to
    be false positives based on simple physical geometry.
    """
    stats = dict(
        enabled=bool(GEOM_CLEAN_EDGES),
        raw_edges=int((df["row_type"] == "edge").sum()),
        raw_divisions=0,
        dropped_nonforward=0,
        dropped_long_jump=0,
        dropped_extra_division_edges=0,
        dropped_bad_division_edges=0,
        final_edges=None,
        final_divisions=None,
    )
    if not GEOM_CLEAN_EDGES:
        e_df = df[df["row_type"] == "edge"]
        stats["raw_divisions"] = int((e_df.groupby(["dataset", "source_id"]).size() >= 2).sum()) if len(e_df) else 0
        stats["final_edges"] = stats["raw_edges"]
        stats["final_divisions"] = stats["raw_divisions"]
        return df, stats

    drop_idx: set[int] = set()
    raw_divisions = 0
    for ds in sorted(df["dataset"].unique()):
        e = _physical_edge_table(df, ds)
        if len(e) == 0:
            continue

        raw_divisions += int((e.groupby("source_id").size() >= 2).sum())

        # Non-forward edges should not represent temporal tracking.
        bad_time = e[e["dt"] <= 0]
        drop_idx.update(bad_time["edge_index"].astype(int).tolist())
        stats["dropped_nonforward"] += int(len(bad_time))

        if DROP_LONG_EDGES:
            too_long = e[(e["dt"] >= 1) & (e["jump_um"] > MAX_EDGE_JUMP_UM)]
            drop_idx.update(too_long["edge_index"].astype(int).tolist())
            stats["dropped_long_jump"] += int(len(too_long))

        if PRUNE_DIVISION_GEOMETRY:
            for source_id, grp in e[e["dt"] >= 1].groupby("source_id"):
                grp = grp.sort_values("jump_um")
                if len(grp) <= 1:
                    continue

                # Keep at most two outgoing edges from a division-like source.
                if len(grp) > DIV_KEEP_TOP_K:
                    extra = grp.iloc[DIV_KEEP_TOP_K:]
                    drop_idx.update(extra["edge_index"].astype(int).tolist())
                    stats["dropped_extra_division_edges"] += int(len(extra))
                    grp = grp.iloc[:DIV_KEEP_TOP_K]

                if len(grp) >= 2:
                    a = grp.iloc[0]
                    b = grp.iloc[1]
                    daughter_sep = float(np.sqrt(
                        ((a["t_z"] - b["t_z"]) * SCALE_UM[0]) ** 2
                        + ((a["t_y"] - b["t_y"]) * SCALE_UM[1]) ** 2
                        + ((a["t_x"] - b["t_x"]) * SCALE_UM[2]) ** 2
                    ))
                    d1 = float(a["jump_um"])
                    d2 = float(b["jump_um"])
                    ratio = d2 / max(d1, 1e-6)

                    # Implausible split: daughters nearly overlap, one daughter is
                    # much farther away, or a child jump is too large.  Demote it
                    # to a single continuation edge by dropping the less plausible
                    # second edge.
                    bad_division = (
                        daughter_sep < DIV_MIN_DAUGHTER_SEP_UM
                        or max(d1, d2) > DIV_MAX_CHILD_JUMP_UM
                        or ratio > DIV_DISTANCE_RATIO_MAX
                    )
                    if bad_division:
                        drop_idx.add(int(b["edge_index"]))
                        stats["dropped_bad_division_edges"] += 1

    stats["raw_divisions"] = int(raw_divisions)
    if drop_idx:
        cleaned = df.drop(index=sorted(drop_idx)).copy()
    else:
        cleaned = df.copy()
    e2 = cleaned[cleaned["row_type"] == "edge"]
    stats["final_edges"] = int(len(e2))
    stats["final_divisions"] = int((e2.groupby(["dataset", "source_id"]).size() >= 2).sum()) if len(e2) else 0
    stats["total_dropped_edges"] = int(len(drop_idx))
    return cleaned, stats


postprocess_stats_by_split: dict[str, dict] = {}

test_names = list_zarr_names(TEST_DIR)
print(f"{len(test_names)} test videos: {test_names[:6]}{'...' if len(test_names)>6 else ''}")

split_rows:  dict[str, list[dict]] = {}
split_stats: list[dict] = []
t_infer = time.time()

for split_name in AVAILABLE_SPLITS:
    wp = REPO_DIR / "weights" / METHOD / split_name / "edge_predictor_best.pth"
    if not wp.exists():
        print(f"  [skip] weights not found: {wp}")
        continue

    print(f"\n=== {split_name} ===")
    t0 = time.time()
    try:
        pred_geffs = run_predict(
            TEST_DIR, test_names, BEST_DET_THR, BEST_DIV_W,
            split_name=split_name,
            tag=f"test_{split_name}",
        )
    except Exception as exc:
        print(f"  FAILED: {exc}")
        continue

    rows: list[dict] = []
    for geff_path in pred_geffs:
        graph = load_pred_graph(geff_path)
        rows.extend(graph_to_rows(graph, geff_path.stem))

    df      = pd.DataFrame(rows)
    df, clean_stats = geometry_cleanup_submission(df)
    postprocess_stats_by_split[split_name] = clean_stats
    if clean_stats.get("enabled"):
        print(
            f"  cleanup: edges {clean_stats['raw_edges']:,}->{clean_stats['final_edges']:,} | "
            f"divisions {clean_stats['raw_divisions']:,}->{clean_stats['final_divisions']:,} | "
            f"dropped={clean_stats.get('total_dropped_edges', 0):,}"
        )
    n_nodes = int((df["row_type"] == "node").sum())
    n_edges = int((df["row_type"] == "edge").sum())
    e_df    = df[df["row_type"] == "edge"]
    n_div   = int((e_df.groupby("source_id").size() >= 2).sum()) if len(e_df) else 0
    elapsed = time.time() - t0

    print(f"  nodes={n_nodes:,}  edges={n_edges:,}  divisions={n_div}  ({elapsed/60:.1f} min)")
    split_rows[split_name]  = df.to_dict("records")
    split_stats.append(dict(
        split=split_name, nodes=n_nodes, edges=n_edges,
        divisions=n_div, minutes=round(elapsed/60, 2), cleanup=postprocess_stats_by_split.get(split_name, {}),
    ))

print(f"\nTotal inference: {(time.time()-t_infer)/60:.1f} min")
if split_stats:
    display(pd.DataFrame(split_stats))

In [ ]:
# Choose the final split with a conservative public-LB heuristic.
# Selecting the split with the most divisions can overfit to false mitosis events.
# We instead prefer a balanced graph: high edge density, but not an extreme number
# of division-like sources. This is intentionally simple and transparent.
if len(split_stats) > 1:
    stat_df = pd.DataFrame(split_stats).copy()
    stat_df["edge_per_node"] = stat_df["edges"] / stat_df["nodes"].clip(lower=1)
    med_div = float(stat_df["divisions"].median())
    stat_df["selection_score"] = (
        stat_df["edge_per_node"]
        - 0.0025 * (stat_df["divisions"] - med_div).abs()
    )
    best_sp = str(stat_df.sort_values("selection_score", ascending=False).iloc[0]["split"])
    print("Split selection table:")
    display(stat_df.sort_values("selection_score", ascending=False))
    print(f"Multiple splits; selecting {best_sp} (balanced edge density / division count)")
elif split_stats:
    best_sp = split_stats[0]["split"]
    print(f"Single split: {best_sp}")
else:
    raise RuntimeError("No successful test inference run.")

SUBMISSION_COLS = [
    "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
]

submission = pd.DataFrame(split_rows[best_sp], columns=SUBMISSION_COLS)
submission.index.name = "id"

# ---------------------------------------------------------------- checks -- #
pred_ds   = set(submission["dataset"].unique())
missing   = sorted(set(test_names) - pred_ds)
if missing:
    raise AssertionError(f"Missing datasets: {missing}")

nodes = submission[submission["row_type"] == "node"]
edges = submission[submission["row_type"] == "edge"]

assert len(nodes) > 0, "No node rows"
assert not submission.isna().any().any(), "NaN in submission"
assert set(submission["row_type"].unique()).issubset({"node", "edge"})
assert (nodes[["node_id","t","z","y","x"]] >= 0).all().all()
assert (nodes[["source_id","target_id"]] == -1).all().all()
if len(edges):
    assert (edges[["node_id","t","z","y","x"]] == -1).all().all()
    assert (edges[["source_id","target_id"]] >= 0).all().all()

for ds, g in submission.groupby("dataset"):
    dn = g[g["row_type"] == "node"]
    de = g[g["row_type"] == "edge"]
    nids = set(dn["node_id"])
    assert dn["node_id"].is_unique,           f"Duplicate node_id in {ds}"
    assert de["source_id"].isin(nids).all(),  f"Dangling source_id in {ds}"
    assert de["target_id"].isin(nids).all(),  f"Dangling target_id in {ds}"

submission.to_csv(SUB_PATH)
print(f"Wrote {SUB_PATH}")
print(f"  {len(submission):,} rows | {len(nodes):,} nodes | {len(edges):,} edges")

# per-dataset breakdown
ds_stats: list[dict] = []
for ds, g in submission.groupby("dataset"):
    gn = g[g["row_type"] == "node"]
    ge = g[g["row_type"] == "edge"]
    od = ge.groupby("source_id").size() if len(ge) else pd.Series(dtype=int)
    ds_stats.append(dict(
        dataset=ds,
        frames=int(gn["t"].nunique()),
        nodes=len(gn),
        edges=len(ge),
        cells_per_frame=round(len(gn) / max(gn["t"].nunique(), 1), 1),
        divisions=int((od >= 2).sum()),
    ))

display(pd.DataFrame(ds_stats))
display(submission.head())

